# Multi-Echelon Stochastic Newsvendor â€” Full Pipeline

> **Bachelor Thesis Project â€” IIT Kharagpur**  
> SRAM-Fused Triton Kernel for Monte-Carlo Inventory Optimization

**Just click `Runtime â†’ Run All` (Colab) or `Cell â†’ Run All` (Jupyter) to execute the entire project end-to-end.**

This notebook:
1. Installs dependencies and clones the codebase from GitHub
2. Builds the data pipeline (M5 topology + tractor/generator distributions)
3. Runs **3 solvers**: CPU-NumPy, PyTorch `torch.compile`, and the fused Triton kernel
4. Validates numerical correctness across all solvers
5. Benchmarks performance (Time / Peak Memory / TFLOPS)
6. Generates visualizations for the thesis
7. **NEW: Runs 4 Newsvendor extensions** (Grid Search Q*, CVaR, Budget-Constrained, Substitution)
8. **NEW: Launches an interactive Gradio app** for the full solver suite

---

## 0. Environment Setup & Clone from GitHub

In [2]:
# â”€â”€ Step 0a: Detect environment â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"Python: {sys.version}")

In [ ]:
# â”€â”€ Step 0b: Install dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
#  torch and triton are pre-installed on Colab GPU runtimes.
#  We install them only if missing (local Jupyter).

def install_if_missing(package, pip_name=None):
    try:
        __import__(package)
        print(f"  {package}: already installed")
    except ImportError:
        name = pip_name or package
        print(f"  {package}: installing {name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])

print("Checking dependencies:")
install_if_missing("torch", "torch")
install_if_missing("triton", "triton")
install_if_missing("numpy")
install_if_missing("pandas")
install_if_missing("matplotlib")
install_if_missing("gradio", "gradio>=4.0")
install_if_missing("plotly", "plotly>=5.0")
print("Done.")

In [4]:
# â”€â”€ Step 0c: Clone the project from GitHub â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
REPO_URL = "https://github.com/Abhijit89Kumar/BTP-2.git"
CLONE_DIR = "BTP-2"

if os.path.isdir(CLONE_DIR):
    print(f"'{CLONE_DIR}/' already exists â€” pulling latest changes ...")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=True)
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, CLONE_DIR], check=True)

# Add to Python path so we can import the modules
project_path = os.path.abspath(CLONE_DIR)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
print(f"Project path: {project_path}")
print(f"Files: {os.listdir(project_path)}")

In [5]:
# â”€â”€ Step 0d: Verify GPU availability â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch

assert torch.cuda.is_available(), (
    "CUDA GPU not found!\n"
    "  - On Colab: Runtime â†’ Change runtime type â†’ GPU (T4/A100)\n"
    "  - Locally: ensure NVIDIA drivers + CUDA toolkit are installed."
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1024**3:.1f} GB")
print(f"SMs: {gpu.multi_processor_count}")
print(f"PyTorch: {torch.__version__}")

import triton
print(f"Triton: {triton.__version__}")

---
## 1. Configuration

All hyperparameters are defined in `config.py`. We use the defaults for the full-scale run:
- **N = 2048** product-location nodes (60% tractors, 40% generators)
- **S = 131,072** Monte-Carlo demand scenarios

In [6]:
from config import NewsvendorConfig, FinancialParams, TritonTuningConfig

cfg = NewsvendorConfig(
    N=2048,           # product-location nodes  (power of 2)
    S=131_072,        # Monte-Carlo scenarios    (2^17)
    seed=42,
    device="cuda",
    dtype=torch.float32,
    tractor_fraction=0.6,
)
fin = FinancialParams()

print("=" * 55)
print("  PROBLEM CONFIGURATION")
print("=" * 55)
print(f"  N (nodes):           {cfg.N:>10,}")
print(f"  S (scenarios):       {cfg.S:>10,}")
print(f"  Tractors:            {cfg.N_tractors:>10,}  ({cfg.tractor_fraction*100:.0f}%)")
print(f"  Generators:          {cfg.N_generators:>10,}  ({(1-cfg.tractor_fraction)*100:.0f}%)")
print(f"  D matrix size:       {cfg.N * cfg.S * 4 / 1024**3:>10.2f} GB  (if materialised)")
print(f"  L matrix size:       {cfg.N * cfg.N * 4 / 1024**2:>10.1f} MB")
print(f"  Device:              {cfg.device:>10}")
print(f"  Dtype:               {'fp32':>10}")
print("=" * 55)

---
## 2. Data Pipeline â€” ETL to GPU Tensors

The pipeline hybridises:
1. **M5 Kaggle Dataset** â€” hierarchical spatial/temporal correlation structure
2. **Tractor/Generator sales distributions** â€” realistic demand parameters
3. **Financial tensors** â€” cost, price, salvage for heavy-machinery margins

In [7]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from data_pipeline import DataPipeline

pipeline = DataPipeline(cfg=cfg, fin=fin)
bundle = pipeline.run()

print("\n" + "=" * 55)
print("  TENSOR BUNDLE (GPU-Resident)")
print("=" * 55)
for name in ["L", "mu", "p", "c", "s", "Q", "Z"]:
    t = getattr(bundle, name)
    mem = t.nelement() * t.element_size() / 1024**2
    print(f"  {name:<4}  shape={str(list(t.shape)):<16}  "
          f"dtype={str(t.dtype):<15}  device={t.device}  mem={mem:>8.2f} MB")
print(f"\n  Tractors:   {bundle.category_mask.sum().item()}")
print(f"  Generators: {(~bundle.category_mask).sum().item()}")
print("=" * 55)

In [ ]:
# â”€â”€ Visualise the data â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Data Pipeline Outputs", fontsize=14, fontweight="bold")

# (a) Cholesky factor L â€” heatmap of top-left corner
ax = axes[0]
L_cpu = bundle.L[:128, :128].cpu().numpy()
im = ax.imshow(L_cpu, cmap="RdBu_r", aspect="auto", vmin=-0.5, vmax=0.5)
ax.set_title("L (Cholesky) â€” top-left 128x128")
ax.set_xlabel("k"); ax.set_ylabel("n")
plt.colorbar(im, ax=ax, fraction=0.046)

# (b) Mean demand distribution
ax = axes[1]
mu_cpu = bundle.mu.squeeze().cpu().numpy()
mask_cpu = bundle.category_mask.cpu().numpy()
ax.hist(mu_cpu[mask_cpu], bins=40, alpha=0.7, label="Tractors", color="#1565C0")
ax.hist(mu_cpu[~mask_cpu], bins=40, alpha=0.7, label="Generators", color="#FF6F00")
ax.set_title("Mean Demand (mu)")
ax.set_xlabel("units/period"); ax.legend()

# (c) Price vs Cost scatter
ax = axes[2]
p_cpu = bundle.p.squeeze().cpu().numpy()
c_cpu = bundle.c.squeeze().cpu().numpy()
ax.scatter(c_cpu[mask_cpu], p_cpu[mask_cpu], s=5, alpha=0.5, label="Tractors", color="#1565C0")
ax.scatter(c_cpu[~mask_cpu], p_cpu[~mask_cpu], s=5, alpha=0.5, label="Generators", color="#FF6F00")
lims = [min(c_cpu.min(), p_cpu.min()), max(c_cpu.max(), p_cpu.max())]
ax.plot(lims, lims, "k--", lw=0.8, label="p = c (break-even)")
ax.set_title("Price vs Cost")
ax.set_xlabel("Unit Cost (c)"); ax.set_ylabel("Selling Price (p)"); ax.legend(fontsize=7)

# (d) Salvage / Cost ratio
ax = axes[3]
s_cpu = bundle.s.squeeze().cpu().numpy()
ratio = s_cpu / c_cpu
ax.hist(ratio[mask_cpu], bins=30, alpha=0.7, label="Tractors", color="#1565C0")
ax.hist(ratio[~mask_cpu], bins=30, alpha=0.7, label="Generators", color="#FF6F00")
ax.set_title("Salvage / Cost Ratio")
ax.set_xlabel("s / c"); ax.legend()

plt.tight_layout()
plt.show()

---
## 3. Solver 1 â€” CPU Baseline (NumPy)

Pure NumPy reference on CPU. Slow, but serves as the **gold-standard** for correctness.

In [9]:
from baseline_solvers import CPUMonteCarlo, SolverResult

print("Running CPU-NumPy solver ...")
print("(This may take 1-3 minutes at full scale N=2048, S=131072)")

cpu_solver = CPUMonteCarlo()
res_cpu = cpu_solver.solve(bundle)

print(f"\nDone: {res_cpu.wall_time_ms:.1f} ms")
print(f"E[profit] range: [{res_cpu.expected_profit.min():.2f}, "
      f"{res_cpu.expected_profit.max():.2f}]")

---
## 4. Solver 2 â€” PyTorch GPU (`torch.compile` + Inductor)

Standard PyTorch with `torch.compile(mode='max-autotune')`.  
The **D[N,S]** intermediate (~1 GB) is materialised in HBM â€” this is the bottleneck our Triton kernel eliminates.

In [ ]:
from baseline_solvers import PyTorchMonteCarlo

print("Running PyTorch-Compile solver (first call triggers JIT) ...")

pt_solver = PyTorchMonteCarlo(use_compile=True)

NUM_REPEATS = 3
best_pt = None
for i in range(NUM_REPEATS):
    res = pt_solver.solve(bundle)
    print(f"  Run {i+1}/{NUM_REPEATS}: {res.wall_time_ms:.2f} ms  "
          f"peak_mem={res.peak_memory_bytes / 1024**3:.3f} GB")
    if best_pt is None or res.wall_time_ms < best_pt.wall_time_ms:
        best_pt = res

res_pt = best_pt
print(f"\nBest: {res_pt.wall_time_ms:.2f} ms  "
      f"peak_mem={res_pt.peak_memory_bytes / 1024**3:.3f} GB")
print(f"E[profit] range: [{res_pt.expected_profit.min():.2f}, "
      f"{res_pt.expected_profit.max():.2f}]")

---
## 5. Solver 3 â€” Triton Fused Kernel (Core Innovation)

The custom `@triton.autotune` + `@triton.jit` kernel **fuses** `L @ Z` matmul with Newsvendor profit logic entirely in SRAM.  
The intermediate **D[N,S]** matrix **never touches HBM**.

In [ ]:
from triton_fused_newsvendor import TritonFusedNewsvendor

print("Running Triton-Fused solver (first call triggers autotune) ...")
print("Autotune will benchmark multiple (BLOCK_M, BLOCK_N, BLOCK_K) configs.")
print("This may take a minute on first run.\n")

tr_solver = TritonFusedNewsvendor()

best_tr = None
for i in range(NUM_REPEATS):
    res = tr_solver.solve(bundle)
    print(f"  Run {i+1}/{NUM_REPEATS}: {res.wall_time_ms:.2f} ms  "
          f"peak_mem={res.peak_memory_bytes / 1024**3:.3f} GB")
    if best_tr is None or res.wall_time_ms < best_tr.wall_time_ms:
        best_tr = res

res_tr = best_tr
print(f"\nBest: {res_tr.wall_time_ms:.2f} ms  "
      f"peak_mem={res_tr.peak_memory_bytes / 1024**3:.3f} GB")
print(f"E[profit] range: [{res_tr.expected_profit.min():.2f}, "
      f"{res_tr.expected_profit.max():.2f}]")

---
## 6. Correctness Validation

Cross-validate all solver pairs with `torch.allclose(atol=1e-2, rtol=1e-3)`.

In [ ]:
from benchmark import check_correctness

print("=" * 65)
print("  CORRECTNESS VALIDATION")
print("=" * 65)
check_correctness(res_cpu, res_pt, atol=1e-2, rtol=1e-3)
check_correctness(res_cpu, res_tr, atol=1e-2, rtol=1e-3)
check_correctness(res_pt, res_tr, atol=1e-2, rtol=1e-3)
print("=" * 65)

---
## 7. Performance Benchmark Table

Compare all 3 solvers: **Execution Time (ms) | Peak Memory (GB) | TFLOPS**

In [ ]:
from benchmark import print_results_table, estimate_flops

all_results = [res_cpu, res_pt, res_tr]
print_results_table(all_results, cfg.N, cfg.S)

---
## 8. Thesis Visualizations

Publication-quality plots for the BTP report.

In [ ]:
# â”€â”€ 8a: Execution Time Comparison Bar Chart â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"Solver Comparison  (N={cfg.N:,}, S={cfg.S:,})",
             fontsize=14, fontweight="bold")

labels = [r.label for r in all_results]
colors = ["#1565C0", "#FF6F00", "#D32F2F"]

# (a) Time
ax = axes[0]
times = [r.wall_time_ms for r in all_results]
bars = ax.bar(labels, times, color=colors, edgecolor="black", linewidth=0.8)
ax.set_ylabel("Execution Time (ms)")
ax.set_title("Wall-Clock Time")
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(times)*0.02,
            f"{t:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, max(times) * 1.15)

# (b) Peak Memory
ax = axes[1]
mems = [r.peak_memory_bytes / 1024**3 for r in all_results]
bars = ax.bar(labels, mems, color=colors, edgecolor="black", linewidth=0.8)
ax.set_ylabel("Peak GPU Memory (GB)")
ax.set_title("Peak Memory Allocated")
for bar, m in zip(bars, mems):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(mems)*0.02,
            f"{m:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, max(mems) * 1.2 if max(mems) > 0 else 1)

# (c) TFLOPS
ax = axes[2]
flops = estimate_flops(cfg.N, cfg.S)
tflops = [flops / (r.wall_time_ms * 1e-3) / 1e12 if r.wall_time_ms > 0 else 0
          for r in all_results]
bars = ax.bar(labels, tflops, color=colors, edgecolor="black", linewidth=0.8)
ax.set_ylabel("TFLOPS")
ax.set_title("Compute Throughput")
for bar, t in zip(bars, tflops):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(tflops)*0.02,
            f"{t:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylim(0, max(tflops) * 1.15 if max(tflops) > 0 else 1)

plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ 8b: Expected Profit Distribution â€” All 3 Solvers Overlaid â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Expected Profit Distribution Across Product Nodes",
             fontsize=14, fontweight="bold")

ep_cpu = res_cpu.expected_profit.cpu().numpy()
ep_pt  = res_pt.expected_profit.cpu().numpy()
ep_tr  = res_tr.expected_profit.cpu().numpy()

# (a) Histogram overlay
ax = axes[0]
ax.hist(ep_cpu, bins=60, alpha=0.5, label="CPU-NumPy", color="#1565C0")
ax.hist(ep_pt,  bins=60, alpha=0.5, label="PyTorch-Compile", color="#FF6F00")
ax.hist(ep_tr,  bins=60, alpha=0.5, label="Triton-Fused", color="#D32F2F")
ax.set_xlabel("Expected Profit per Node")
ax.set_ylabel("Count")
ax.set_title("Profit Distribution (all 3 solvers)")
ax.legend()

# (b) Parity scatter: Triton vs CPU
ax = axes[1]
ax.scatter(ep_cpu, ep_tr, s=3, alpha=0.4, c="#D32F2F")
lims = [min(ep_cpu.min(), ep_tr.min()), max(ep_cpu.max(), ep_tr.max())]
ax.plot(lims, lims, "k--", lw=1, label="Perfect agreement")
ax.set_xlabel("CPU-NumPy E[profit]")
ax.set_ylabel("Triton-Fused E[profit]")
ax.set_title("Numerical Parity: Triton vs CPU")
ax.legend()
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

# Print summary stats
diff = np.abs(ep_cpu - ep_tr)
print(f"Max absolute diff (CPU vs Triton): {diff.max():.6f}")
print(f"Mean absolute diff:                {diff.mean():.6f}")
print(f"Relative error (mean):             {(diff / (np.abs(ep_cpu) + 1e-8)).mean():.2e}")

In [ ]:
# â”€â”€ 8c: Per-Category Profit Analysis â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

mask_cpu_np = bundle.category_mask.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Expected Profit by Inventory Category (Triton Kernel Results)",
             fontsize=14, fontweight="bold")

# Tractors
ax = axes[0]
ax.hist(ep_tr[mask_cpu_np], bins=40, color="#1565C0", edgecolor="black", linewidth=0.5)
ax.axvline(ep_tr[mask_cpu_np].mean(), color="red", linestyle="--",
           label=f"Mean = {ep_tr[mask_cpu_np].mean():.2f}")
ax.set_title(f"Tractors  (N = {mask_cpu_np.sum()})")
ax.set_xlabel("E[Profit]")
ax.legend()

# Generators
ax = axes[1]
ax.hist(ep_tr[~mask_cpu_np], bins=40, color="#FF6F00", edgecolor="black", linewidth=0.5)
ax.axvline(ep_tr[~mask_cpu_np].mean(), color="red", linestyle="--",
           label=f"Mean = {ep_tr[~mask_cpu_np].mean():.2f}")
ax.set_title(f"Generators  (N = {(~mask_cpu_np).sum()})")
ax.set_xlabel("E[Profit]")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ 8d: Memory Savings Waterfall â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title("HBM Memory Traffic Comparison", fontsize=14, fontweight="bold")

D_size_gb = cfg.N * cfg.S * 4 / 1024**3
L_size_gb = cfg.N * cfg.N * 4 / 1024**3
Z_size_gb = cfg.N * cfg.S * 4 / 1024**3
out_size_gb = cfg.N * 4 / 1024**3

# PyTorch: reads L, Z; writes D; reads D for min/profit; writes Profit; reads for mean
pt_traffic = L_size_gb + Z_size_gb + 3 * D_size_gb  # conservative estimate
# Triton: reads tiles of L, Z; writes only out[N]
tr_traffic = L_size_gb + Z_size_gb + out_size_gb

categories = ["PyTorch\n(torch.compile)", "Triton\n(fused kernel)"]
traffic = [pt_traffic, tr_traffic]
bars = ax.bar(categories, traffic, color=["#FF6F00", "#D32F2F"],
              edgecolor="black", linewidth=0.8, width=0.5)

for bar, t in zip(bars, traffic):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{t:.2f} GB", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylabel("Estimated HBM Traffic (GB)")
savings = (1 - tr_traffic / pt_traffic) * 100
ax.annotate(f"{savings:.0f}% reduction",
            xy=(1, tr_traffic), xytext=(1.3, pt_traffic * 0.6),
            fontsize=12, fontweight="bold", color="#2E7D32",
            arrowprops=dict(arrowstyle="->", color="#2E7D32", lw=2))

ax.set_ylim(0, pt_traffic * 1.25)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"PyTorch HBM traffic (est.):  {pt_traffic:.2f} GB")
print(f"Triton HBM traffic (est.):   {tr_traffic:.2f} GB")
print(f"Savings: {savings:.1f}%")
print(f"D matrix eliminated: {D_size_gb:.2f} GB")

---
## 9. Scaling Sweep â€” Performance Across Problem Sizes

Sweep **N** from 128 to 2048 (scenarios fixed at 65,536) to show how the Triton kernel scales.

In [ ]:
# â”€â”€ Scaling sweep â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

S_SWEEP = 65_536
N_VALUES = [128, 256, 512, 1024, 2048]

sweep_results = {"N": [], "PyTorch_ms": [], "Triton_ms": [],
                 "PyTorch_mem_GB": [], "Triton_mem_GB": []}

print(f"{'N':>6} | {'PyTorch (ms)':>14} | {'Triton (ms)':>14} | {'Speedup':>8}")
print("-" * 52)

for N_val in N_VALUES:
    sweep_cfg = NewsvendorConfig(N=N_val, S=S_SWEEP, device="cuda", seed=42)
    sweep_bundle = DataPipeline(cfg=sweep_cfg, fin=fin).run()

    # PyTorch
    pt = PyTorchMonteCarlo(use_compile=True)
    pt.solve(sweep_bundle)  # warmup
    rp = pt.solve(sweep_bundle)

    # Triton
    tr = TritonFusedNewsvendor()
    tr.solve(sweep_bundle)  # warmup (includes autotune)
    rt = tr.solve(sweep_bundle)

    speedup = rp.wall_time_ms / rt.wall_time_ms if rt.wall_time_ms > 0 else 0
    print(f"{N_val:>6} | {rp.wall_time_ms:>14.2f} | {rt.wall_time_ms:>14.2f} | {speedup:>7.1f}x")

    sweep_results["N"].append(N_val)
    sweep_results["PyTorch_ms"].append(rp.wall_time_ms)
    sweep_results["Triton_ms"].append(rt.wall_time_ms)
    sweep_results["PyTorch_mem_GB"].append(rp.peak_memory_bytes / 1024**3)
    sweep_results["Triton_mem_GB"].append(rt.peak_memory_bytes / 1024**3)

    # Free GPU memory
    del sweep_bundle, rp, rt
    torch.cuda.empty_cache()

print("-" * 52)

In [ ]:
# â”€â”€ Scaling plots â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"Scaling Sweep  (S = {S_SWEEP:,} fixed)",
             fontsize=14, fontweight="bold")

Ns = sweep_results["N"]

# (a) Execution time
ax = axes[0]
ax.plot(Ns, sweep_results["PyTorch_ms"], "o-", color="#FF6F00",
        label="PyTorch-Compile", linewidth=2, markersize=7)
ax.plot(Ns, sweep_results["Triton_ms"], "s-", color="#D32F2F",
        label="Triton-Fused", linewidth=2, markersize=7)
ax.set_xlabel("N (product-location nodes)")
ax.set_ylabel("Execution Time (ms)")
ax.set_title("Wall-Clock Time vs N")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xscale("log", base=2)

# (b) Speedup
ax = axes[1]
speedups = [p / t if t > 0 else 0
            for p, t in zip(sweep_results["PyTorch_ms"], sweep_results["Triton_ms"])]
ax.bar([str(n) for n in Ns], speedups, color="#2E7D32",
       edgecolor="black", linewidth=0.8)
for i, (n, s) in enumerate(zip(Ns, speedups)):
    ax.text(i, s + 0.05, f"{s:.1f}x", ha="center", fontsize=10, fontweight="bold")
ax.set_xlabel("N")
ax.set_ylabel("Speedup (Triton / PyTorch)")
ax.set_title("Speedup vs N")
ax.set_ylim(0, max(speedups) * 1.2 if speedups else 1)
ax.grid(axis="y", alpha=0.3)

# (c) Peak memory
ax = axes[2]
ax.plot(Ns, sweep_results["PyTorch_mem_GB"], "o-", color="#FF6F00",
        label="PyTorch-Compile", linewidth=2, markersize=7)
ax.plot(Ns, sweep_results["Triton_mem_GB"], "s-", color="#D32F2F",
        label="Triton-Fused", linewidth=2, markersize=7)
ax.set_xlabel("N (product-location nodes)")
ax.set_ylabel("Peak Memory (GB)")
ax.set_title("Peak GPU Memory vs N")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xscale("log", base=2)

plt.tight_layout()
plt.show()

---
## 10. Summary & Key Findings

In [ ]:
# â”€â”€ Final summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

flops_full = estimate_flops(cfg.N, cfg.S)

speedup_vs_cpu = res_cpu.wall_time_ms / res_tr.wall_time_ms if res_tr.wall_time_ms > 0 else 0
speedup_vs_pt  = res_pt.wall_time_ms / res_tr.wall_time_ms if res_tr.wall_time_ms > 0 else 0
triton_tflops  = flops_full / (res_tr.wall_time_ms * 1e-3) / 1e12 if res_tr.wall_time_ms > 0 else 0

diff_cpu_tr = (res_cpu.expected_profit.cpu() - res_tr.expected_profit.cpu()).abs()

print("=" * 65)
print("  FINAL RESULTS SUMMARY")
print("=" * 65)
print(f"")
print(f"  Problem Scale:")
print(f"    N = {cfg.N:,}  nodes  |  S = {cfg.S:,}  scenarios")
print(f"    D matrix (eliminated): {cfg.N * cfg.S * 4 / 1024**3:.2f} GB")
print(f"")
print(f"  Performance:")
print(f"    CPU-NumPy:        {res_cpu.wall_time_ms:>10.1f} ms")
print(f"    PyTorch-Compile:  {res_pt.wall_time_ms:>10.2f} ms")
print(f"    Triton-Fused:     {res_tr.wall_time_ms:>10.2f} ms")
print(f"")
print(f"  Speedup (Triton vs CPU):     {speedup_vs_cpu:>8.1f}x")
print(f"  Speedup (Triton vs PyTorch): {speedup_vs_pt:>8.1f}x")
print(f"  Triton Throughput:           {triton_tflops:>8.2f} TFLOPS")
print(f"")
print(f"  Memory:")
print(f"    PyTorch peak:  {res_pt.peak_memory_bytes / 1024**3:.3f} GB")
print(f"    Triton  peak:  {res_tr.peak_memory_bytes / 1024**3:.3f} GB")
mem_savings = (1 - res_tr.peak_memory_bytes / res_pt.peak_memory_bytes) * 100 if res_pt.peak_memory_bytes > 0 else 0
print(f"    Reduction:     {mem_savings:.1f}%")
print(f"")
print(f"  Correctness (CPU vs Triton):")
print(f"    Max |diff|:    {diff_cpu_tr.max().item():.6f}")
print(f"    Mean |diff|:   {diff_cpu_tr.mean().item():.6f}")
print(f"")
print(f"  GPU: {torch.cuda.get_device_properties(0).name}")
print("=" * 65)
print("\nAll cells executed successfully. Project complete.")

---
## 11. Newsvendor Extensions

Four extensions beyond the base newsvendor, each with its own **Triton GPU kernel**:

| Extension | Description | Kernel Innovation |
|-----------|-------------|-------------------|
| **Grid Search Q*** | Find optimal order quantity per product over K grid points | Matmul once, K-loop business logic in SRAM |
| **CVaR (Risk-Averse)** | Average profit in worst Î±% of scenarios | Fused profit + histogram in single kernel |
| **Budget-Constrained** | Total procurement budget Î£ cÂ·Q â‰¤ B | Lagrangian bisection reusing grid search kernel |
| **Substitution** | Demand redirected when products stock out | Two-pass kernel: stockout â†’ adjusted profit |

### 11a. Extension 1 — Optimal Q* Grid Search

Find the **optimal order quantity Q*** for each product by evaluating expected profit at K grid points.  
The Triton kernel performs the expensive matmul **once**, then loops over K Q-values entirely in SRAM.

In [ ]:
from config import GridSearchConfig
from extensions.grid_search_solvers import CPUGridSearch, PyTorchGridSearch, TritonGridSearch

grid_cfg = GridSearchConfig(K=64, q_ratio_min=0.3, q_ratio_max=2.5)

print("=" * 65)
print("  EXTENSION 1: OPTIMAL Q* GRID SEARCH")
print(f"  K={grid_cfg.K} grid points, Q ratio range=[{grid_cfg.q_ratio_min}, {grid_cfg.q_ratio_max}]")
print("=" * 65)

# CPU baseline
print("\n[CPU-GridSearch] Running ...")
gs_cpu = CPUGridSearch().solve(bundle, grid_cfg)
print(f"  Time: {gs_cpu.wall_time_ms:.1f} ms")
print(f"  Q* range: [{gs_cpu.Q_star.min():.2f}, {gs_cpu.Q_star.max():.2f}]")
print(f"  Best E[profit] range: [{gs_cpu.best_profit.min():.2f}, {gs_cpu.best_profit.max():.2f}]")

# PyTorch GPU
print("\n[PyTorch-GridSearch] Running ...")
gs_pt = PyTorchGridSearch(use_compile=True).solve(bundle, grid_cfg)
print(f"  Time: {gs_pt.wall_time_ms:.2f} ms")

# Triton GPU
print("\n[Triton-GridSearch] Running (first call triggers autotune) ...")
gs_tr = TritonGridSearch().solve(bundle, grid_cfg)
print(f"  Time: {gs_tr.wall_time_ms:.2f} ms")
print(f"  Peak mem: {gs_tr.peak_memory_bytes / 1024**3:.3f} GB")

# Correctness
diff = (gs_cpu.best_profit - gs_tr.best_profit.cpu()).abs()
print(f"\nMax |best_profit| diff (CPU vs Triton): {diff.max().item():.6f}")

if gs_pt.wall_time_ms > 0:
    print(f"Speedup Triton vs PyTorch: {gs_pt.wall_time_ms / gs_tr.wall_time_ms:.1f}x")

# Q* improvement over base
base_profit = res_tr.expected_profit.cpu().float()
gs_profit = gs_tr.best_profit.cpu().float()
improvement = (gs_profit - base_profit).mean().item()
print(f"\nMean profit improvement Q* vs Q=mu: {improvement:.2f} ({improvement/base_profit.abs().mean().item()*100:.1f}%)")

### 11b. Extension 2 — CVaR (Conditional Value at Risk)

**Risk-averse** newsvendor: CVaR at level α is the average profit in the **worst α%** of scenarios.  
The Triton kernel computes both expected profit AND a per-product profit histogram in a single fused pass.

In [ ]:
from config import CVaRConfig
from extensions.cvar_solvers import CPUCVaR, PyTorchCVaR, TritonCVaR

cvar_cfg = CVaRConfig(alpha=0.05, num_bins=256)

print("=" * 65)
print("  EXTENSION 2: CVaR RISK-AVERSE NEWSVENDOR")
print(f"  Alpha = {cvar_cfg.alpha} (worst {cvar_cfg.alpha*100:.0f}% of scenarios)")
print("=" * 65)

print("\n[CPU-CVaR] Running ...")
cvar_cpu = CPUCVaR(cvar_cfg).solve(bundle)
print(f"  Time: {cvar_cpu.wall_time_ms:.1f} ms")
print(f"  VaR range: [{cvar_cpu.VaR.min():.2f}, {cvar_cpu.VaR.max():.2f}]")
print(f"  CVaR range: [{cvar_cpu.CVaR.min():.2f}, {cvar_cpu.CVaR.max():.2f}]")

print("\n[PyTorch-CVaR] Running ...")
cvar_pt = PyTorchCVaR(cvar_cfg, use_compile=True).solve(bundle)
print(f"  Time: {cvar_pt.wall_time_ms:.2f} ms")

print("\n[Triton-CVaR] Running ...")
cvar_tr = TritonCVaR(cvar_cfg).solve(bundle)
print(f"  Time: {cvar_tr.wall_time_ms:.2f} ms")
print(f"  Peak mem: {cvar_tr.peak_memory_bytes / 1024**3:.3f} GB")

cvar_check = (cvar_cpu.CVaR <= cvar_cpu.expected_profit + 0.01).all()
print(f"\nSanity check CVaR <= E[profit]: {'PASS' if cvar_check else 'FAIL'}")

diff_ep = (cvar_cpu.expected_profit - cvar_tr.expected_profit.cpu()).abs()
print(f"Max |E[profit]| diff (CPU vs Triton): {diff_ep.max().item():.6f}")
if cvar_pt.wall_time_ms > 0:
    print(f"Speedup Triton vs PyTorch: {cvar_pt.wall_time_ms / cvar_tr.wall_time_ms:.1f}x")

### 11c. Extension 3 — Budget-Constrained Newsvendor

Total procurement budget: **Σ c_i · Q_i ≤ B**. Solved via **Lagrangian dual bisection** — each iteration reuses the Grid Search Triton kernel with modified cost c' = c·(1+λ).

In [ ]:
from config import BudgetConfig
from extensions.budget_solvers import CPUBudget, TritonBudget, _compute_budget_default

budget_cfg = BudgetConfig(budget_fraction=0.7, max_iterations=25, tolerance=1e-3)
grid_cfg_budget = GridSearchConfig(K=64, q_ratio_min=0.3, q_ratio_max=2.5)
B = _compute_budget_default(bundle, budget_cfg)

print("=" * 65)
print("  EXTENSION 3: BUDGET-CONSTRAINED NEWSVENDOR")
print(f"  Budget B = {B:,.0f} (70% of unconstrained cost)")
print("=" * 65)

print("\n[CPU-Budget] Running ...")
budget_cpu = CPUBudget().solve(bundle, grid_cfg_budget, budget_cfg)
print(f"  Time: {budget_cpu.wall_time_ms:.1f} ms | Iterations: {len(budget_cpu.lambda_history)}")
print(f"  lambda* = {budget_cpu.lambda_star:.4f} | Cost: {budget_cpu.total_cost:,.0f} / {budget_cpu.budget:,.0f}")

print("\n[Triton-Budget] Running ...")
budget_tr = TritonBudget().solve(bundle, grid_cfg_budget, budget_cfg)
print(f"  Time: {budget_tr.wall_time_ms:.2f} ms | Iterations: {len(budget_tr.lambda_history)}")
print(f"  lambda* = {budget_tr.lambda_star:.4f} | Cost: {budget_tr.total_cost:,.0f} / {budget_tr.budget:,.0f}")

if budget_cpu.wall_time_ms > 0:
    print(f"\nSpeedup Triton vs CPU: {budget_cpu.wall_time_ms / budget_tr.wall_time_ms:.1f}x")

### 11d. Extension 4 — Multi-Product Substitution

When product i stocks out, a fraction β of unmet demand is **redirected** to substitute products.  
Uses a **two-pass Triton kernel**: Pass 1 computes stockout → HBM; Pass 2 loads neighbors + computes adjusted profit.

In [ ]:
from config import SubstitutionConfig
from data_pipeline import SubstitutionGraphGenerator
from extensions.substitution_solvers import CPUSubstitution, PyTorchSubstitution, TritonSubstitution

sub_cfg = SubstitutionConfig(max_subs=4, beta_min=0.05, beta_max=0.30)
mask_np = bundle.category_mask.cpu().numpy()
sub_idx_np, sub_frac_np = SubstitutionGraphGenerator(sub_cfg).generate(bundle.N, mask_np, seed=cfg.seed)
sub_idx = torch.tensor(sub_idx_np, dtype=torch.long, device=cfg.device)
sub_frac = torch.tensor(sub_frac_np, dtype=torch.float32, device=cfg.device)

print("=" * 65)
print("  EXTENSION 4: MULTI-PRODUCT SUBSTITUTION")
print(f"  Max substitutes: {sub_cfg.max_subs}, Beta range: [{sub_cfg.beta_min}, {sub_cfg.beta_max}]")
print("=" * 65)

print("\n[CPU-Substitution] Running ...")
sub_cpu = CPUSubstitution().solve(bundle, sub_idx_np, sub_frac_np)
print(f"  Time: {sub_cpu.wall_time_ms:.1f} ms")
print(f"  Mean redirected demand: {sub_cpu.substitution_demand.mean():.4f}")

print("\n[PyTorch-Substitution] Running ...")
sub_pt = PyTorchSubstitution(use_compile=True).solve(bundle, sub_idx, sub_frac)
print(f"  Time: {sub_pt.wall_time_ms:.2f} ms")

print("\n[Triton-Substitution] Running ...")
sub_tr = TritonSubstitution().solve(bundle, sub_idx, sub_frac, max_subs=sub_cfg.max_subs)
print(f"  Time: {sub_tr.wall_time_ms:.2f} ms")
print(f"  Peak mem: {sub_tr.peak_memory_bytes / 1024**3:.3f} GB")

diff = (sub_cpu.expected_profit - sub_tr.expected_profit.cpu()).abs()
print(f"\nMax |E[profit]| diff (CPU vs Triton): {diff.max().item():.6f}")
if sub_pt.wall_time_ms > 0:
    print(f"Speedup Triton vs PyTorch: {sub_pt.wall_time_ms / sub_tr.wall_time_ms:.1f}x")

### 11e. Extensions Performance Summary

In [ ]:
print("=" * 75)
print("  EXTENSIONS PERFORMANCE SUMMARY")
print("=" * 75)
print(f"{'Extension':<25} | {'CPU (ms)':>12} | {'PyTorch (ms)':>14} | {'Triton (ms)':>13} | {'Speedup':>8}")
print("-" * 75)

rows = [
    ("Grid Search Q*", gs_cpu.wall_time_ms, gs_pt.wall_time_ms, gs_tr.wall_time_ms),
    ("CVaR Risk-Averse", cvar_cpu.wall_time_ms, cvar_pt.wall_time_ms, cvar_tr.wall_time_ms),
    ("Budget-Constrained", budget_cpu.wall_time_ms, 0, budget_tr.wall_time_ms),
    ("Substitution", sub_cpu.wall_time_ms, sub_pt.wall_time_ms, sub_tr.wall_time_ms),
]

for name, cpu_ms, pt_ms, tr_ms in rows:
    speedup = cpu_ms / tr_ms if tr_ms > 0 else 0
    pt_str = f"{pt_ms:.2f}" if pt_ms > 0 else "N/A"
    print(f"{name:<25} | {cpu_ms:>12.1f} | {pt_str:>14} | {tr_ms:>13.2f} | {speedup:>7.1f}x")

print("-" * 75)
print("\nAll 4 extensions executed successfully!")

---
## 12. Interactive Gradio App

Launch an **interactive web app** to solve all 5 newsvendor variants with 3 solver backends.

The app provides:
- **Problem Setup**: configure N, S, seed, VRAM estimation
- **Solver Configuration**: select variant, tune parameters, choose backends
- **Results Dashboard**: Plotly performance charts, variant-specific visualizations
- **Per-Product Analysis**: drill into individual products
- **About**: mathematical formulations & architecture

> **Note**: On Colab, this creates a **public share link** you can open in any browser.

In [ ]:
# ── Launch the Gradio Newsvendor Solver Suite ────────────────────────────
from gradio_app.app import launch

# On Colab, share=True creates a public URL (valid for 72 hours)
launch(share=True)